# Gradient Boosting 칼날 마모 예측 (Blade Wear Prediction)
슈레더 칼날 마모율과 잔여 수명(RUL)을 Gradient Boosting으로 예측하는 프로덕션 모델

## Step 0. 라이브러리 설치 및 임포트

In [ ]:
!pip install -q scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

print("라이브러리 로드 완료")

## Step 1. 데이터 생성 (3 Blade Cycles x 365일)

슈레더 14종 센서 데이터 생성기를 내장하여 3번의 칼날 교체 주기 데이터를 생성합니다.
- 1 사이클 = 365일, 10분 간격 = ~52,560 샘플
- 3 사이클 총 **~157,680 샘플**
- 각 사이클마다 칼날 마모율이 0%에서 100%까지 상승 후 리셋

In [ ]:
# ──────────────────────────────────────────────
# 슈레더 센서 데이터 생성기 (common_data.py 내장)
# ──────────────────────────────────────────────

# 상수 정의
_BASE_RPM_A = 1200.0      # A축 정격 RPM
_BASE_RPM_B = 800.0       # B축 정격 RPM
_BASE_CUR_A = 85.0        # A축 정격 전류 (A)
_BASE_CUR_B = 60.0        # B축 정격 전류 (A)
_BASE_VIB = 2.5           # 정상 진동 RMS (mm/s)
_BASE_TEMP = 35.0         # 정상 베어링 온도
_BASE_IR_TEMP = 45.0      # 정상 IR 표면 온도
_BASE_DUST = 5.0          # 정상 분진 농도 (mg/m3)
_BASE_GAS_VOC = 10.0      # 정상 VOC (ppm)
_BASE_GAS_H2 = 2.0        # 정상 H2 (ppm)
_BASE_GAS_CO = 3.0        # 정상 CO (ppm)
_BASE_WEIGHT = 150.0      # 10분당 처리량 (kg)

# 이상 이벤트 확률
_P_BEARING_ANOMALY = 0.003
_P_FIRE_EVENT = 0.001
_P_DUST_SPIKE = 0.005
_P_GAS_LEAK = 0.002
_P_JAM_EVENT = 0.004


def _operating_mask(timestamps):
    """가동 마스크: 평일 08-18시 정상가동, 야간/주말 대기/정지"""
    hours = np.array([ts.hour for ts in timestamps])
    dow = np.array([ts.dayofweek for ts in timestamps])
    operating = np.ones(len(timestamps), dtype=float)
    operating[dow >= 5] = 0.0
    night_mask = (hours < 8) | (hours >= 18)
    operating[night_mask & (dow < 5)] = 0.15
    lunch_mask = (hours == 12)
    operating[lunch_mask & (dow < 5)] = 0.6
    return operating


def _daily_cycle(hours, phase_shift=0.0):
    """일간 사인파 패턴"""
    return np.sin(2 * np.pi * hours / 24 - np.pi / 2 + phase_shift)


def _wear_trend(n_points, days, max_wear_pct=30.0):
    """칼날 마모 트렌드 (비선형 가속)"""
    t_norm = np.linspace(0, 1, n_points)
    wear = max_wear_pct * t_norm ** 1.3
    return wear


def _inject_events(n_points, rng, event_prob, duration_range=(3, 15)):
    """이벤트 마스크 생성"""
    mask = np.zeros(n_points, dtype=bool)
    intensity = np.zeros(n_points)
    i = 0
    while i < n_points:
        if rng.random() < event_prob:
            dur = rng.integers(duration_range[0], duration_range[1] + 1)
            end = min(i + dur, n_points)
            mask[i:end] = True
            event_len = end - i
            peak = rng.uniform(0.5, 1.0)
            ramp = np.concatenate([
                np.linspace(0, peak, event_len // 2 + 1),
                np.linspace(peak, 0, event_len - event_len // 2)
            ])[:event_len]
            intensity[i:end] = ramp
            i = end + rng.integers(50, 200)
        else:
            i += 1
    return mask, intensity


def generate_shredder_full_data(days=365, freq_minutes=10, seed=42):
    """슈레더 전체 센서 데이터 시뮬레이션 (14종 센서 + 칼날 마모율)"""
    rng = np.random.default_rng(seed)
    n_points = days * 24 * 60 // freq_minutes
    timestamps = pd.date_range(
        start='2026-01-01', periods=n_points, freq=f'{freq_minutes}min'
    )

    hours = np.array([ts.hour for ts in timestamps])
    t = np.arange(n_points)

    # 가동 마스크
    op = _operating_mask(timestamps)

    # 칼날 마모 트렌드
    wear = _wear_trend(n_points, days, max_wear_pct=30.0)
    wear_factor = 1.0 + wear / 100.0

    # 이상 이벤트
    bearing_mask, bearing_int = _inject_events(n_points, rng, _P_BEARING_ANOMALY, (5, 20))
    fire_mask, fire_int = _inject_events(n_points, rng, _P_FIRE_EVENT, (3, 10))
    dust_mask, dust_int = _inject_events(n_points, rng, _P_DUST_SPIKE, (5, 25))
    gas_mask, gas_int = _inject_events(n_points, rng, _P_GAS_LEAK, (10, 40))
    jam_mask, jam_int = _inject_events(n_points, rng, _P_JAM_EVENT, (2, 8))

    # 일간 사이클
    daily = _daily_cycle(hours)
    daily_shifted = _daily_cycle(hours, phase_shift=np.pi / 6)

    # SPD: 모터 속도
    spd_a = (_BASE_RPM_A * op + 30.0 * daily * op
             + rng.normal(0, 8, n_points) * op - 400.0 * jam_int * op)
    spd_a = np.clip(spd_a, 0, _BASE_RPM_A * 1.15)
    spd_b = (_BASE_RPM_B * op + 20.0 * daily * op
             + rng.normal(0, 6, n_points) * op - 300.0 * jam_int * op)
    spd_b = np.clip(spd_b, 0, _BASE_RPM_B * 1.15)

    # CUR: 모터 전류
    cur_a = (_BASE_CUR_A * op * wear_factor + 8.0 * daily * op
             + rng.normal(0, 2.0, n_points) * op + 25.0 * jam_int * op)
    cur_a = np.clip(cur_a, 0, 180)
    cur_b = (_BASE_CUR_B * op * wear_factor + 5.0 * daily * op
             + rng.normal(0, 1.5, n_points) * op + 18.0 * jam_int * op)
    cur_b = np.clip(cur_b, 0, 130)

    # VIB: 3축 진동
    def _make_vib(base, axis_weight, bearing_scale):
        vib = (base * op * wear_factor * axis_weight
               + 0.4 * daily * op * axis_weight
               + rng.normal(0, 0.25, n_points) * op
               + bearing_scale * bearing_int * op
               + 1.5 * jam_int * op)
        return np.clip(vib, 0, 30)

    vib_a_x = _make_vib(_BASE_VIB, 1.0, 12.0)
    vib_a_y = _make_vib(_BASE_VIB, 0.8, 10.0)
    vib_a_z = _make_vib(_BASE_VIB, 0.5, 6.0)
    vib_b_x = _make_vib(_BASE_VIB * 0.8, 1.0, 9.0)
    vib_b_y = _make_vib(_BASE_VIB * 0.8, 0.8, 7.5)
    vib_b_z = _make_vib(_BASE_VIB * 0.8, 0.5, 4.5)

    # TMP: 온도
    tmp_ir1 = (_BASE_IR_TEMP * np.maximum(op, 0.4) + 5.0 * daily * op
               + wear * 0.1 * op + rng.normal(0, 1.2, n_points)
               + 80.0 * fire_int + 8.0 * jam_int * op)
    tmp_ir2 = tmp_ir1 + rng.normal(0, 1.5, n_points) - 2.0
    tmp_ir1 = np.clip(tmp_ir1, 15, 350)
    tmp_ir2 = np.clip(tmp_ir2, 15, 340)

    tmp_a = (_BASE_TEMP * np.maximum(op, 0.5) + 3.0 * daily_shifted * op
             + wear * 0.08 * op + rng.normal(0, 0.6, n_points)
             + 15.0 * bearing_int * op + 30.0 * fire_int)
    tmp_a = np.clip(tmp_a, 15, 150)
    tmp_b = (_BASE_TEMP * 0.9 * np.maximum(op, 0.5) + 2.5 * daily_shifted * op
             + wear * 0.06 * op + rng.normal(0, 0.5, n_points)
             + 12.0 * bearing_int * op + 25.0 * fire_int)
    tmp_b = np.clip(tmp_b, 15, 140)

    # DST: 분진
    dst_1 = (_BASE_DUST * op * wear_factor + 1.5 * daily * op
             + rng.normal(0, 0.8, n_points) * op
             + rng.exponential(0.5, n_points) * op
             + 40.0 * dust_int * op + 5.0 * jam_int * op)
    dst_1 = np.clip(dst_1, 0, 80)

    # GAS: 가스
    gas_voc = (_BASE_GAS_VOC * np.maximum(op, 0.3) + 2.0 * daily * op
               + rng.normal(0, 1.0, n_points) + 200.0 * gas_int + 15.0 * fire_int)
    gas_voc = np.clip(gas_voc, 0, 500)
    gas_h2 = (_BASE_GAS_H2 * np.maximum(op, 0.2) + 0.3 * daily * op
              + rng.normal(0, 0.3, n_points) + 80.0 * gas_int + 20.0 * fire_int)
    gas_h2 = np.clip(gas_h2, 0, 200)
    gas_co = (_BASE_GAS_CO * np.maximum(op, 0.2) + 0.5 * daily * op
              + rng.normal(0, 0.4, n_points) + 50.0 * gas_int + 40.0 * fire_int)
    gas_co = np.clip(gas_co, 0, 200)

    # SCL: 처리량
    scl_weight = (_BASE_WEIGHT * op + 15.0 * daily * op
                  + rng.normal(0, 5.0, n_points) * op
                  - 80.0 * jam_int * op - wear * 0.3 * op)
    scl_weight = np.clip(scl_weight, 0, 250)

    # 이벤트 라벨
    event_labels = np.full(n_points, 'normal', dtype=object)
    event_labels[bearing_mask] = 'bearing_anomaly'
    event_labels[fire_mask] = 'fire_event'
    event_labels[dust_mask] = 'dust_spike'
    event_labels[gas_mask] = 'gas_leak'
    event_labels[jam_mask] = 'jam_event'

    df = pd.DataFrame({
        'timestamp': timestamps,
        'VIB_A_x': np.round(vib_a_x, 3), 'VIB_A_y': np.round(vib_a_y, 3),
        'VIB_A_z': np.round(vib_a_z, 3), 'VIB_B_x': np.round(vib_b_x, 3),
        'VIB_B_y': np.round(vib_b_y, 3), 'VIB_B_z': np.round(vib_b_z, 3),
        'CUR_A': np.round(cur_a, 2), 'CUR_B': np.round(cur_b, 2),
        'SPD_A': np.round(spd_a, 1), 'SPD_B': np.round(spd_b, 1),
        'TMP_IR1': np.round(tmp_ir1, 1), 'TMP_IR2': np.round(tmp_ir2, 1),
        'TMP_A': np.round(tmp_a, 1), 'TMP_B': np.round(tmp_b, 1),
        'DST_1': np.round(dst_1, 2), 'GAS_VOC': np.round(gas_voc, 1),
        'GAS_H2': np.round(gas_h2, 1), 'GAS_CO': np.round(gas_co, 1),
        'SCL_weight': np.round(scl_weight, 1),
        'blade_wear_pct': np.round(wear, 2),
        'event_label': event_labels,
    })
    return df

print("데이터 생성기 정의 완료")

In [ ]:
%%time
# 3 사이클 데이터 생성 (각 사이클 365일, 서로 다른 시드)
cycles = []
for cycle_id in range(1, 4):
    print(f"사이클 {cycle_id}/3 생성 중... (seed={40 + cycle_id})")
    df_cycle = generate_shredder_full_data(days=365, freq_minutes=10, seed=40 + cycle_id)

    # 칼날 마모율: 0% -> 100% (원래 0->30%를 풀 사이클로 스케일링)
    df_cycle['blade_wear_pct'] = np.round(
        df_cycle['blade_wear_pct'] / 30.0 * 100.0, 2
    )
    df_cycle['cycle'] = cycle_id

    # 타임스탬프 조정 (연속적으로)
    if cycle_id > 1:
        offset = cycles[-1]['timestamp'].iloc[-1] + pd.Timedelta(minutes=10)
        time_delta = offset - df_cycle['timestamp'].iloc[0]
        df_cycle['timestamp'] = df_cycle['timestamp'] + time_delta

    cycles.append(df_cycle)

df = pd.concat(cycles, ignore_index=True)

print(f"\n{'='*50}")
print(f"총 데이터 볼륨")
print(f"{'='*50}")
print(f"총 샘플 수: {len(df):,}")
print(f"사이클 수: {df['cycle'].nunique()}")
print(f"기간: {df['timestamp'].min()} ~ {df['timestamp'].max()}")
print(f"컬럼 수: {df.shape[1]}")
print(f"메모리 사용량: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\n사이클별 샘플 수:")
print(df['cycle'].value_counts().sort_index())
df.head(3)

## Step 2. 원시 데이터 시각화

6채널 센서 개요, 칼날 마모 3사이클 추이, 센서-마모 상관관계를 확인합니다.

In [ ]:
# 2-1. 6채널 센서 개요 (서브샘플링하여 시각화)

# VIB RMS 계산
df['VIB_RMS_A'] = np.sqrt(df['VIB_A_x']**2 + df['VIB_A_y']**2 + df['VIB_A_z']**2)
df['VIB_RMS_B'] = np.sqrt(df['VIB_B_x']**2 + df['VIB_B_y']**2 + df['VIB_B_z']**2)

sample_idx = np.arange(0, len(df), 50)  # 시각화용 다운샘플
df_vis = df.iloc[sample_idx]

fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharex=True)

channels_all = [
    ('CUR_A', 'Current A (A)', 'tab:blue'),
    ('CUR_B', 'Current B (A)', 'tab:cyan'),
    ('SPD_A', 'Speed A (RPM)', 'tab:red'),
    ('SPD_B', 'Speed B (RPM)', 'tab:orange'),
    ('VIB_RMS_A', 'Vibration RMS A (mm/s)', 'tab:green'),
    ('VIB_RMS_B', 'Vibration RMS B (mm/s)', 'tab:olive'),
]

for ax, (col, label, color) in zip(axes.flat, channels_all):
    ax.plot(df_vis['timestamp'], df_vis[col], color=color, alpha=0.7, linewidth=0.5)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
    for c in [1, 2, 3]:
        cycle_data = df[df['cycle'] == c]
        if c > 1:
            ax.axvline(cycle_data['timestamp'].iloc[0], color='red',
                       linestyle='--', alpha=0.5, linewidth=1)

axes[0, 0].set_title('6-Channel Sensor Overview (3 Blade Cycles)')
axes[-1, 0].set_xlabel('Timestamp')
axes[-1, 1].set_xlabel('Timestamp')
plt.tight_layout()
plt.show()
print("6채널 센서 시계열 시각화 완료")

In [ ]:
# 2-2. 칼날 마모 3사이클 추이
fig, ax = plt.subplots(figsize=(16, 4))
colors_cycle = ['tab:blue', 'tab:orange', 'tab:green']
for c in [1, 2, 3]:
    mask = df['cycle'] == c
    ax.plot(df.loc[mask, 'timestamp'].values[::20],
            df.loc[mask, 'blade_wear_pct'].values[::20],
            color=colors_cycle[c-1], label=f'Cycle {c}', linewidth=1.5)

ax.axhline(80, color='orange', linestyle='--', alpha=0.7, label='Warning (80%)')
ax.axhline(95, color='red', linestyle='--', alpha=0.7, label='Critical (95%)')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Blade Wear (%)')
ax.set_title('Blade Wear Progression Over 3 Cycles')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("칼날 마모 3사이클 추이 시각화 완료")

In [ ]:
# 2-3. 센서-마모 상관관계 히트맵
sensor_cols = ['CUR_A', 'CUR_B', 'SPD_A', 'SPD_B', 'VIB_RMS_A', 'VIB_RMS_B',
               'TMP_IR1', 'TMP_A', 'DST_1', 'GAS_VOC', 'SCL_weight', 'blade_wear_pct']
corr = df[sensor_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(sensor_cols)))
ax.set_yticks(range(len(sensor_cols)))
ax.set_xticklabels(sensor_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(sensor_cols, fontsize=9)
for i in range(len(sensor_cols)):
    for j in range(len(sensor_cols)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7,
                color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Sensor-Wear Correlation Matrix')
plt.tight_layout()
plt.show()
print("센서-마모 상관관계 분석 완료")

## Step 3. 특성 엔지니어링 (24 Features)

6개 채널 (CUR_A, CUR_B, SPD_A, SPD_B, VIB_RMS_A, VIB_RMS_B) x 4개 통계량 (mean, std, max, min) = **24개 특성**

롤링 윈도우 크기 = 6 (60분 = 1시간 윈도우)

In [ ]:
%%time
# 6채널 x 4통계 = 24 특성 생성
base_channels = ['CUR_A', 'CUR_B', 'SPD_A', 'SPD_B', 'VIB_RMS_A', 'VIB_RMS_B']
stats = ['mean', 'std', 'max', 'min']
window = 6  # 60분 (10분 x 6)

feature_names = []
for ch in base_channels:
    rolling = df[ch].rolling(window=window, min_periods=1)
    for stat in stats:
        feat_name = f'{ch}_roll_{stat}'
        feature_names.append(feat_name)
        if stat == 'mean':
            df[feat_name] = rolling.mean()
        elif stat == 'std':
            df[feat_name] = rolling.std().fillna(0)
        elif stat == 'max':
            df[feat_name] = rolling.max()
        elif stat == 'min':
            df[feat_name] = rolling.min()

print(f"생성된 특성 수: {len(feature_names)}")
print(f"\n특성 목록 ({len(feature_names)}개):")
for i, fn in enumerate(feature_names, 1):
    print(f"  {i:2d}. {fn}")

print(f"\n특성 데이터 shape: {df[feature_names].shape}")
print(f"결측치 수: {df[feature_names].isnull().sum().sum()}")

## Step 4. 타겟 변수: 마모율(%) 및 잔여 수명(RUL)

- **blade_wear_pct**: 현재 칼날 마모율 (0~100%)
- **RUL (Remaining Useful Life)**: 마모 100% 도달까지 남은 시간 (hours)

In [ ]:
# RUL 계산: 각 사이클 내에서 100% 도달까지 남은 시간 (hours)
df['RUL_hours'] = 0.0
for c in df['cycle'].unique():
    mask = df['cycle'] == c
    cycle_len = mask.sum()
    remaining_points = np.arange(cycle_len, 0, -1)
    df.loc[mask, 'RUL_hours'] = np.round(remaining_points * 10 / 60, 2)

print("타겟 변수 요약:")
print(f"\n  blade_wear_pct:")
print(f"    범위: {df['blade_wear_pct'].min():.1f}% ~ {df['blade_wear_pct'].max():.1f}%")
print(f"    평균: {df['blade_wear_pct'].mean():.1f}%")

print(f"\n  RUL (hours):")
print(f"    범위: {df['RUL_hours'].min():.1f}h ~ {df['RUL_hours'].max():.1f}h")
print(f"    평균: {df['RUL_hours'].mean():.1f}h")
print(f"    = {df['RUL_hours'].max() / 24:.0f}일 (최대)")

## Step 5. 학습/테스트 분할

- **학습 데이터**: 사이클 1 + 사이클 2 (~85%)
- **테스트 데이터**: 사이클 3 (~15%)

시간 순서를 유지하는 분할로 미래 데이터 누출을 방지합니다.

In [ ]:
# 사이클 기반 분할: Cycle 1+2 = Train, Cycle 3 = Test
train_mask = df['cycle'].isin([1, 2])
test_mask = df['cycle'] == 3

X_train = df.loc[train_mask, feature_names].values
X_test = df.loc[test_mask, feature_names].values

y_train_wear = df.loc[train_mask, 'blade_wear_pct'].values
y_test_wear = df.loc[test_mask, 'blade_wear_pct'].values

y_train_rul = df.loc[train_mask, 'RUL_hours'].values
y_test_rul = df.loc[test_mask, 'RUL_hours'].values

print(f"학습/테스트 분할 완료:")
print(f"  학습 세트: {X_train.shape[0]:,} 샘플 ({X_train.shape[0]/len(df)*100:.1f}%)")
print(f"  테스트 세트: {X_test.shape[0]:,} 샘플 ({X_test.shape[0]/len(df)*100:.1f}%)")
print(f"  특성 수: {X_train.shape[1]}")
print(f"\n학습 마모율 범위: {y_train_wear.min():.1f}% ~ {y_train_wear.max():.1f}%")
print(f"테스트 마모율 범위: {y_test_wear.min():.1f}% ~ {y_test_wear.max():.1f}%")

## Step 6. 모델 학습

Gradient Boosting Regressor 2개 모델:
1. **마모율 예측 모델** (Wear %)
2. **잔여 수명 예측 모델** (RUL hours)

하이퍼파라미터: `n_estimators=300, max_depth=6, learning_rate=0.05`

In [ ]:
%%time
# 모델 1: 마모율 예측
print("모델 1: Blade Wear % 예측 모델 학습 중...")
model_wear = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
    verbose=0
)
model_wear.fit(X_train, y_train_wear)
print("  마모율 모델 학습 완료")

# 모델 2: RUL 예측
print("\n모델 2: RUL (Remaining Useful Life) 예측 모델 학습 중...")
model_rul = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
    verbose=0
)
model_rul.fit(X_train, y_train_rul)
print("  RUL 모델 학습 완료")

# 예측
y_pred_wear = model_wear.predict(X_test)
y_pred_rul = model_rul.predict(X_test)

print(f"\n예측 완료: 테스트 세트 {len(y_pred_wear):,} 샘플")

## Step 7. 모델 평가

MAE, RMSE, R2 지표로 마모율 및 RUL 예측 성능을 평가합니다.

In [ ]:
# 성능 평가
def evaluate_model(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

results = []
results.append(evaluate_model(y_test_wear, y_pred_wear, 'Wear %'))
results.append(evaluate_model(y_test_rul, y_pred_rul, 'RUL (hours)'))

df_results = pd.DataFrame(results)

print("=" * 60)
print("모델 성능 평가 결과 (테스트 세트 - Cycle 3)")
print("=" * 60)
for _, row in df_results.iterrows():
    print(f"\n  [{row['Model']}]")
    print(f"    MAE  = {row['MAE']:.4f}")
    print(f"    RMSE = {row['RMSE']:.4f}")
    print(f"    R2   = {row['R2']:.4f}")
print("\n" + "=" * 60)

# 학습 세트 성능 확인 (과적합 체크)
y_train_pred_wear = model_wear.predict(X_train)
y_train_pred_rul = model_rul.predict(X_train)

print("\n과적합 체크 (학습 세트 R2):")
print(f"  Wear % - Train R2: {r2_score(y_train_wear, y_train_pred_wear):.4f}, Test R2: {r2_score(y_test_wear, y_pred_wear):.4f}")
print(f"  RUL    - Train R2: {r2_score(y_train_rul, y_train_pred_rul):.4f}, Test R2: {r2_score(y_test_rul, y_pred_rul):.4f}")

## Step 8. 시각화

### 8-1. 마모율 예측 vs 실측

In [ ]:
# 8-1. Wear % Prediction vs Actual
test_timestamps = df.loc[test_mask, 'timestamp'].values
plot_idx = np.arange(0, len(y_test_wear), 10)

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(test_timestamps[plot_idx], y_test_wear[plot_idx],
        color='tab:blue', alpha=0.8, linewidth=1.2, label='Actual')
ax.plot(test_timestamps[plot_idx], y_pred_wear[plot_idx],
        color='tab:red', alpha=0.7, linewidth=1.0, linestyle='--', label='Predicted')
ax.fill_between(test_timestamps[plot_idx],
                y_pred_wear[plot_idx] - 2, y_pred_wear[plot_idx] + 2,
                alpha=0.15, color='red', label='Prediction Band (+/-2%)')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Blade Wear (%)')
ax.set_title('Blade Wear Prediction vs Actual (Test Set - Cycle 3)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("마모율 예측 vs 실측 시각화 완료")

### 8-2. Feature Importance Top 15

In [ ]:
# 8-2. Feature Importance Top 15
importances = model_wear.feature_importances_
feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(feat_imp['Feature'], feat_imp['Importance'],
               color='steelblue', edgecolor='navy', alpha=0.8)
ax.set_xlabel('Feature Importance')
ax.set_title('Top 15 Feature Importance (Wear % Model)')
ax.grid(True, axis='x', alpha=0.3)

for bar, val in zip(bars, feat_imp['Importance']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()
print("특성 중요도 Top 15 시각화 완료")

### 8-3. RUL 예측

In [ ]:
# 8-3. RUL Prediction vs Actual
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(test_timestamps[plot_idx], y_test_rul[plot_idx],
        color='tab:blue', alpha=0.8, linewidth=1.2, label='Actual RUL')
ax.plot(test_timestamps[plot_idx], y_pred_rul[plot_idx],
        color='tab:red', alpha=0.7, linewidth=1.0, linestyle='--', label='Predicted RUL')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Remaining Useful Life (hours)')
ax.set_title('RUL Prediction vs Actual (Test Set - Cycle 3)')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("RUL 예측 시각화 완료")

### 8-4. 오차 분포

In [ ]:
# 8-4. Error Distribution
errors_wear = y_pred_wear - y_test_wear
errors_rul = y_pred_rul - y_test_rul

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(errors_wear, bins=80, color='steelblue', alpha=0.7, edgecolor='navy')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].axvline(np.mean(errors_wear), color='orange', linestyle='-', linewidth=1.5,
                label=f'Mean = {np.mean(errors_wear):.3f}')
axes[0].set_xlabel('Prediction Error (%)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Wear % Prediction Error Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(errors_rul, bins=80, color='coral', alpha=0.7, edgecolor='darkred')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].axvline(np.mean(errors_rul), color='orange', linestyle='-', linewidth=1.5,
                label=f'Mean = {np.mean(errors_rul):.1f}h')
axes[1].set_xlabel('Prediction Error (hours)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('RUL Prediction Error Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("오차 분포 시각화 완료")

### 8-5. Actual vs Predicted 산점도

In [ ]:
# 8-5. Actual vs Predicted Scatter Plot
scatter_idx = np.random.default_rng(42).choice(
    len(y_test_wear), size=5000, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(y_test_wear[scatter_idx], y_pred_wear[scatter_idx],
                alpha=0.3, s=8, c='steelblue', edgecolors='none')
lim_w = [0, max(y_test_wear.max(), y_pred_wear.max()) * 1.05]
axes[0].plot(lim_w, lim_w, 'r--', linewidth=1.5, label='Perfect Prediction')
axes[0].set_xlabel('Actual Wear (%)')
axes[0].set_ylabel('Predicted Wear (%)')
axes[0].set_title('Actual vs Predicted - Wear %')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal')

axes[1].scatter(y_test_rul[scatter_idx], y_pred_rul[scatter_idx],
                alpha=0.3, s=8, c='coral', edgecolors='none')
lim_r = [0, max(y_test_rul.max(), y_pred_rul.max()) * 1.05]
axes[1].plot(lim_r, lim_r, 'r--', linewidth=1.5, label='Perfect Prediction')
axes[1].set_xlabel('Actual RUL (hours)')
axes[1].set_ylabel('Predicted RUL (hours)')
axes[1].set_title('Actual vs Predicted - RUL')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()
print("Actual vs Predicted 산점도 시각화 완료")

### 8-6. 마모 임계치 알림 (80%, 90%, 95%)

In [ ]:
# 8-6. Wear Threshold Alerts (80%, 90%, 95%)
thresholds = [
    (80, 'Warning', 'orange'),
    (90, 'Danger', 'orangered'),
    (95, 'Critical', 'red'),
]

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(test_timestamps[plot_idx], y_pred_wear[plot_idx],
        color='tab:blue', linewidth=1.2, label='Predicted Wear %', zorder=3)

ax.axhspan(0, 80, alpha=0.08, color='green', label='Normal Zone (0-80%)')
ax.axhspan(80, 90, alpha=0.12, color='orange', label='Warning Zone (80-90%)')
ax.axhspan(90, 95, alpha=0.15, color='orangered', label='Danger Zone (90-95%)')
ax.axhspan(95, 105, alpha=0.2, color='red', label='Critical Zone (95-100%)')

for thresh, name, color in thresholds:
    ax.axhline(thresh, color=color, linestyle='--', linewidth=1.5, alpha=0.8)
    ax.text(test_timestamps[plot_idx[-1]], thresh + 0.8,
            f'  {name} ({thresh}%)', color=color, fontsize=10, fontweight='bold')

for thresh, name, color in thresholds:
    exceed_idx = np.where(y_pred_wear > thresh)[0]
    if len(exceed_idx) > 0:
        first_exceed = exceed_idx[0]
        ax.axvline(test_timestamps[first_exceed], color=color, linestyle=':',
                   alpha=0.6, linewidth=1)
        ax.scatter([test_timestamps[first_exceed]], [y_pred_wear[first_exceed]],
                   color=color, s=80, zorder=5, edgecolors='black', linewidths=0.5)

ax.set_xlabel('Timestamp')
ax.set_ylabel('Predicted Blade Wear (%)')
ax.set_title('Blade Wear Threshold Alert System')
ax.set_ylim(0, 105)
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("칼날 마모 알림 요약 (예측값 기준):")
for thresh, name, color in thresholds:
    exceed_idx = np.where(y_pred_wear > thresh)[0]
    if len(exceed_idx) > 0:
        first_ts = test_timestamps[exceed_idx[0]]
        total_above = len(exceed_idx)
        pct_above = total_above / len(y_pred_wear) * 100
        print(f"  [{name} {thresh}%] 최초 도달: {pd.Timestamp(first_ts).strftime('%Y-%m-%d %H:%M')} | "
              f"초과 샘플: {total_above:,} ({pct_above:.1f}%)")
    else:
        print(f"  [{name} {thresh}%] 도달하지 않음")

## Step 9. 요약

### 모델 개요
| 항목 | 내용 |
|------|------|
| **알고리즘** | Gradient Boosting Regressor (scikit-learn) |
| **타겟 변수** | 칼날 마모율 (%), 잔여 수명 RUL (hours) |
| **입력 특성** | 6채널 x 4통계 = 24개 롤링 특성 |
| **채널** | CUR_A, CUR_B, SPD_A, SPD_B, VIB_RMS_A, VIB_RMS_B |
| **학습 데이터** | 사이클 1+2 (~105,120 샘플) |
| **테스트 데이터** | 사이클 3 (~52,560 샘플) |
| **하이퍼파라미터** | n_estimators=300, max_depth=6, lr=0.05 |

### 핵심 결과
- 칼날 마모율과 RUL을 Gradient Boosting으로 예측
- 3사이클 (1,095일) 프로덕션 규모 데이터로 학습
- 시간 기반 분할로 미래 데이터 누출 방지
- 80/90/95% 임계치 알림 시스템으로 예방 정비 지원

### 활용 방안
1. **예방 정비**: 마모 80% 도달 예측 시 교체 일정 수립
2. **긴급 대응**: 95% 도달 예측 시 즉시 교체 알림
3. **비용 최적화**: RUL 기반 칼날 수명 최대 활용
4. **가동률 향상**: 예측 기반 계획 정비로 비계획 정지 최소화